# 04 - YAML Configuration

In addition to the class-based API, orthograph supports YAML configuration files for defining graph data models. This is useful for configuration-driven workflows, sharing schemas across teams, or defining schemas without writing Python code.

This notebook covers:
- Defining node and relationship types in YAML
- Loading YAML from strings and files
- Property types, required vs optional fields
- Cardinality constraints in YAML
- Round-tripping between Python models and YAML

In [2]:
from orthograph import (
    GraphDataModel,
    GraphValidator,
    NodeModel,
    RelationshipModel,
    Cardinality,
)
from orthograph.io.yaml import load_yaml_string, load_yaml_file, save_yaml_file
from pathlib import Path
import tempfile

## Defining a model in YAML

A YAML model definition has three sections:
- `name` -- the model name
- `node_types` -- a mapping of label to node specification
- `relationship_types` -- a mapping of label to relationship specification

Each node type specifies `uid_field` (optional), and `properties` (a mapping of property name to type spec). Each relationship type specifies `source`, `target`, and optionally `properties`, `directed`, and cardinality.

In [3]:
filmography_yaml = """
name: Filmography
version: "1.0"

node_types:
  Person:
    uid_field: name
    properties:
      name:
        type: str
        required: true
      born:
        type: int
        required: false

  Movie:
    uid_field: title
    properties:
      title:
        type: str
        required: true
      year:
        type: int
        required: true

relationship_types:
  ACTED_IN:
    source: Person
    target: Movie
    directed: true
    properties:
      role:
        type: str
        required: true

  DIRECTED:
    source: Person
    target: Movie
    directed: true
"""

model = load_yaml_string(filmography_yaml)

print("Model name: ", model.name)
print("Node labels:", model.node_labels)
print("Rel labels: ", model.relationship_labels)

Model name:  Filmography
Node labels: {'Person', 'Movie'}
Rel labels:  {'DIRECTED', 'ACTED_IN'}


## YAML property types

The `type` field in a property spec maps to Python types:

| YAML type | Python type |
|-----------|-------------|
| `str`     | `str`       |
| `int`     | `int`       |
| `float`   | `float`     |
| `bool`    | `bool`      |
| `list`    | `list`      |
| `dict`    | `dict`      |

Properties are **required** by default. Set `required: false` to make a property optional (the field becomes `Optional[T]` with a `None` default).

In [4]:
# Inspect the Person node type -- 'name' is required, 'born' is optional
person_type = model.get_node_type("Person")

print("Required properties:", person_type.get_required_property_names())
print("All properties:     ", person_type.get_all_property_names())
print()

# Inspect the Movie node type -- both properties are required
movie_type = model.get_node_type("Movie")

print("Required properties:", movie_type.get_required_property_names())
print("All properties:     ", movie_type.get_all_property_names())

Required properties: {'name'}
All properties:      {'name', 'born'}

Required properties: {'year', 'title'}
All properties:      {'year', 'title'}


## Cardinality in YAML

Relationship types can define `source_cardinality` and `target_cardinality` to constrain how many relationships of that type a node can have.

The YAML syntax uses `min` and `max` keys:

```yaml
source_cardinality:
  min: 0
  max: 1       # null or omitted means unbounded
target_cardinality:
  min: 1
  max: null     # one or more
```

If cardinality is not specified, the default is `0..N` (zero or more).

In [5]:
cardinality_yaml = """
name: CardinalityExample

node_types:
  Author:
    uid_field: name
    properties:
      name:
        type: str
        required: true

  Book:
    uid_field: isbn
    properties:
      isbn:
        type: str
        required: true
      title:
        type: str
        required: true

relationship_types:
  WROTE:
    source: Author
    target: Book
    directed: true
    source_cardinality:
      min: 0
      max: null
    target_cardinality:
      min: 1
      max: null
"""

card_model = load_yaml_string(cardinality_yaml)
wrote_type = card_model.get_relationship_type("WROTE")

print("WROTE source cardinality:", repr(wrote_type.__source_cardinality__))
print("WROTE target cardinality:", repr(wrote_type.__target_cardinality__))
print()
print("An Author can have 0..N outgoing WROTE relationships.")
print("A Book must have 1..N incoming WROTE relationships (at least one author).")

WROTE source cardinality: CardinalitySpec(0..N)
WROTE target cardinality: CardinalitySpec(1..N)

An Author can have 0..N outgoing WROTE relationships.
A Book must have 1..N incoming WROTE relationships (at least one author).


## Saving a Python model to YAML

You can also go the other direction: define a model in Python using classes, then export it to YAML. This round-trip is useful for generating configuration files from code, or for migrating between the two approaches.

In [6]:
from typing import Optional


# Define a small model in Python
class City(NodeModel):
    __label__ = "City"
    __uid_field__ = "name"
    name: str
    population: Optional[int] = None


class Country(NodeModel):
    __label__ = "Country"
    __uid_field__ = "code"
    code: str
    name: str


class LocatedIn(RelationshipModel):
    __label__ = "LOCATED_IN"
    __source_type__ = City
    __target_type__ = Country
    __directed__ = True
    __source_cardinality__ = Cardinality.ONE  # a city is in exactly one country


py_model = GraphDataModel(
    name="Geography",
    version="0.1",
    node_types=[City, Country],
    relationship_types=[LocatedIn],
)

# Write to a temp file, then read back the YAML content
tmp = Path(tempfile.mktemp(suffix=".yaml"))
save_yaml_file(py_model, tmp)

print("--- Generated YAML ---")
print(tmp.read_text(encoding="utf-8"))

# Load it back and verify
loaded_model = load_yaml_file(tmp)
print("Loaded model name:", loaded_model.name)
print("Node labels match: ", loaded_model.node_labels == py_model.node_labels)
print("Rel labels match:  ", loaded_model.relationship_labels == py_model.relationship_labels)

# Clean up
tmp.unlink()

--- Generated YAML ---
name: Geography
version: '0.1'
node_types:
  City:
    uid_field: name
    properties:
      name:
        type: str
        required: true
      population:
        type: int
        required: false
  Country:
    uid_field: code
    properties:
      code:
        type: str
        required: true
      name:
        type: str
        required: true
relationship_types:
  LOCATED_IN:
    source: City
    target: Country
    directed: true
    source_cardinality:
      min: 1
      max: 1
    target_cardinality:
      min: 0
      max: null

Loaded model name: Geography
Node labels match:  True
Rel labels match:   True


## Loading from a file

In production workflows, you would typically store the YAML definition in a file and load it at application startup. Here we write a file, load it, and use the resulting model with `GraphValidator`.

In [7]:
# Write a YAML file to disk
config_yaml = """
name: SupplyChain

node_types:
  Warehouse:
    uid_field: code
    properties:
      code:
        type: str
        required: true
      city:
        type: str
        required: true

  Product:
    uid_field: sku
    properties:
      sku:
        type: str
        required: true
      name:
        type: str
        required: true
      weight_kg:
        type: float
        required: false

relationship_types:
  STORES:
    source: Warehouse
    target: Product
    directed: true
    properties:
      quantity:
        type: int
        required: true
"""

config_path = Path(tempfile.mktemp(suffix=".yaml"))
config_path.write_text(config_yaml.strip(), encoding="utf-8")

# Load from file
supply_model = load_yaml_file(config_path)
print("Model:", supply_model.name)
print("Nodes:", supply_model.node_labels)
print("Rels: ", supply_model.relationship_labels)

# Validate some data against the loaded model
validator = GraphValidator(supply_model)

nodes = [
    {"__label__": "Warehouse", "code": "WH-01", "city": "Berlin"},
    {"__label__": "Product", "sku": "SKU-100", "name": "Widget", "weight_kg": 0.5},
]
rels = [
    {"__label__": "STORES", "__source_uid__": "WH-01", "__target_uid__": "SKU-100", "quantity": 250},
]

result = validator.validate(nodes=nodes, relationships=rels)
print("\nValid:", result.is_valid)

# Clean up
config_path.unlink()

Model: SupplyChain
Nodes: {'Warehouse', 'Product'}
Rels:  {'STORES'}

Valid: True


## Validating data against a YAML-defined model

End-to-end workflow: define a model in YAML, load it, create a validator, and validate both valid and invalid data.

In [9]:
# Full end-to-end workflow
model = load_yaml_string(filmography_yaml)
validator = GraphValidator(model)

In [10]:
# --- Valid data ---
valid_nodes = [
    {"__label__": "Person", "name": "Keanu Reeves", "born": 1964},
    {"__label__": "Person", "name": "Lana Wachowski"},
    {"__label__": "Movie", "title": "The Matrix", "year": 1999},
]
valid_rels = [
    {"__label__": "ACTED_IN", "__source_uid__": "Keanu Reeves", "__target_uid__": "The Matrix", "role": "Neo"},
    {"__label__": "DIRECTED", "__source_uid__": "Lana Wachowski", "__target_uid__": "The Matrix"},
]

result = validator.validate(nodes=valid_nodes, relationships=valid_rels)
print("Valid data result:", result.is_valid)
print("Errors:", len(result.errors))
print()


Valid data result: True
Errors: 0



In [11]:
# --- Invalid data: missing required property, unknown label ---
invalid_nodes = [
    {"__label__": "Person", "name": "Alice"},
    {"__label__": "Movie", "title": "Film X"},          # missing required 'year'
    {"__label__": "Studio", "name": "Warner Bros"},       # unknown label
]

result = validator.validate_nodes(nodes=invalid_nodes)
print("Invalid data result:", result.is_valid)
print("Errors found:")
for issue in result.errors:
    print(f"  [{issue.code}] {issue.message}")

Invalid data result: False
Errors found:
  [PROPERTY_VALIDATION_ERROR] Validation error: Field required (field: year)
  [UNKNOWN_NODE_LABEL] Unknown node label: Studio
